# Register an Azure ML Model

Register a small transparent JSON model as an immutable custom-model asset and retrieve its metadata.

**Source:** Adapted from [Azure/azureml-examples model.ipynb](https://github.com/Azure/azureml-examples/blob/70f0ddd0cf92fb54af4031db0bb1dbd4a8443543/sdk/python/assets/model/model.ipynb), MIT License.

In [1]:
from pathlib import Path
import json
import os

from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Model
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
MODEL_NAME = os.environ["WORKSHOP_MODEL_NAME"]
REGISTER = os.getenv("REGISTER_FOUNDATION_MODEL", "false").lower() in {"1", "true", "yes"}

In [2]:
model_dir = WORKSHOP_ROOT / "data/models/demo"
model_document = json.loads((model_dir / "model.json").read_text(encoding="utf-8"))

model_definition = Model(
    name=MODEL_NAME,
    type=AssetTypes.CUSTOM_MODEL,
    path=str(model_dir),
    description="Transparent two-feature taxi baseline used by the Azure ML workshop",
    tags={
        "workshop": "azureml-h2o",
        "format": "json-linear-model",
        "features": ",".join(model_document["features"]),
    },
)

if REGISTER:
    registered_model = ml_client.models.create_or_update(model_definition)
    verified_model = ml_client.models.get(
        MODEL_NAME,
        version=registered_model.version,
    )
    assert verified_model.name == MODEL_NAME
    assert verified_model.version == registered_model.version
    print(f"Registered model: {verified_model.name}:{verified_model.version}")
else:
    print(f"Prepared model definition: {MODEL_NAME} (version assigned on registration)")
    print("Registration disabled. Set REGISTER_FOUNDATION_MODEL=true in workshop/.env.")

Registered model: workshop-taxi-model:3


## Expected Result

The transparent JSON model is registered as an immutable custom-model asset with workshop provenance tags.

Next: `05_submit_command_job.ipynb`.